# Isotropic plate with an offset

For a homogeneous isotropic plate of thickness $h$ whose mid-surface is at
$z = d$ from the reference surface, i.e. `offset = d`, the thickness integrals
have the closed form

$$
A = h C, \qquad B = d h C, \qquad D = h \left( d^2 + \frac{h^2}{12}
\right) C, \qquad A_{ts} = \frac{5}{6} G h I
$$

where $C$ is the plane-stress stiffness of the material and $I$ the 2x2
identity matrix. The offset creates a membrane-bending coupling $B \neq 0$
and, in the same way, the odd moments of the density, see
[Mass integrals](../mass_integrals.rst). The code below, the unit test
`test_example_isotropic_plate_offset` of `tests/test_core.py`, checks these
expressions, the matrices $E$, $F$, $H$ of the TSDT and the six mass integrals
against the integrals $\int_{d - h/2}^{d + h/2} z^n dz$.

In [1]:
import sys
sys.path.insert(0, '../tests') # helpers of the tests

import numpy as np

from composites.utils import isotropic_plate

In [2]:
# closed form of an isotropic plate whose mid-surface is at z = d
E, nu, h, d, rho = 70e9, 0.3, 2e-3, 0.6e-3, 2700.
lam = isotropic_plate(thickness=h, E=E, nu=nu, offset=d, rho=rho)
C = E/(1 - nu**2)*np.array([[1, nu, 0], [nu, 1, 0], [0, 0, (1 - nu)/2]])
G = E/(2*(1 + nu))
atol = 1e-12*E
assert np.allclose(lam.A, h*C, rtol=1e-12, atol=atol*h)
assert np.allclose(lam.B, d*h*C, rtol=1e-12, atol=atol*h*d)
assert np.allclose(lam.D, h*(d**2 + h**2/12)*C, rtol=1e-12, atol=atol*h**3)
# the shear correction of an isotropic plate is 5/6
assert np.allclose(lam.Ats, 5/6*G*h*np.eye(2), rtol=1e-12, atol=atol*h)
print('B =\n', lam.B)
print('d h C =\n', d*h*C)
print('Ats/(G h) =\n', lam.Ats/(G*h))

B =
 [[92307.69230769 27692.30769231     0.        ]
 [27692.30769231 92307.69230769     0.        ]
 [    0.             0.         32307.69230769]]
d h C =
 [[92307.69230769 27692.30769231     0.        ]
 [27692.30769231 92307.69230769     0.        ]
 [    0.             0.         32307.69230769]]
Ats/(G h) =
 [[ 0.83333333 -0.        ]
 [-0.          0.83333333]]


In [3]:
# with s uniform in [-h/2, h/2]: <s^2> = h^2/12, <s^4> = h^4/80,
# <s^6> = h^6/448, and int (d + s)^n ds = h <(d + s)^n>
m = [1., d, d**2 + h**2/12, d**3 + d*h**2/4,
     d**4 + d**2*h**2/2 + h**4/80, None,
     d**6 + 15/12*d**4*h**2 + 15/80*d**2*h**4 + h**6/448]
assert np.allclose(lam.E, h*m[3]*C, rtol=1e-12, atol=atol*h**4)
assert np.allclose(lam.F, h*m[4]*C, rtol=1e-12, atol=atol*h**5)
assert np.allclose(lam.H, h*m[6]*C, rtol=1e-12, atol=atol*h**7)
for name, n in (('intrho', 0), ('intrhoz', 1), ('intrhoz2', 2),
                ('intrhoz3', 3), ('intrhoz4', 4), ('intrhoz6', 6)):
    assert np.isclose(getattr(lam, name), rho*h*m[n], rtol=1e-12, atol=0)
    print('%-8s = %.6e, closed form %.6e' % (name, getattr(lam, name), rho*h*m[n]))

intrho   = 5.400000e+00, closed form 5.400000e+00
intrhoz  = 3.240000e-03, closed form 3.240000e-03
intrhoz2 = 3.744000e-06, closed form 3.744000e-06
intrhoz3 = 4.406400e-09, closed form 4.406400e-09
intrhoz4 = 5.667840e-12, closed form 5.667840e-12
intrhoz6 = 1.035457e-17, closed form 1.035457e-17


In [4]:
# without offset the odd moments vanish
lam0 = isotropic_plate(thickness=h, E=E, nu=nu, rho=rho)
assert abs(lam0.intrhoz) < 1e-15*rho*h**2
assert abs(lam0.intrhoz3) < 1e-15*rho*h**4
assert np.allclose(lam0.B, 0, atol=atol*h**2)
print('intrhoz, intrhoz3 =', lam0.intrhoz, lam0.intrhoz3)

intrhoz, intrhoz3 = 0.0 0.0
